# LP Optimization Validation

Validates the end-to-end LP optimization layer (`optimization/run_lp_optimization.py`).

Covers five scenarios:
1. **Baseline** — default params, single product, all facilities
2. **Diversification-constrained** — `max_supplier_share=0.40`
3. **Budget infeasibility** — budget cap set below minimum feasible cost
4. **Service-level buffer** — `service_level_target=1.10` (+10% procurement buffer)
5. **Facility filter** — restrict to a single facility

# Imports

In [1]:
import pandas as pd
import sys
import psycopg2
import os
from pathlib import Path

# move up from analytics/analysis/ to repo root
PROJECT_ROOT = Path.cwd().resolve().parents[1]
while PROJECT_ROOT.name != "procurement_agent" and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT))
print(PROJECT_ROOT)

from dotenv import load_dotenv
from optimization.run_lp_optimization import LPParams, run

/Users/jonathanbeck/Library/CloudStorage/OneDrive-Personal/Desktop/GWU_Spring_26/Business Analytics Capstone/procurement_agent


# DB Connection

In [2]:
load_dotenv()

True

In [3]:
conn = psycopg2.connect(
    dbname=os.getenv('PGDATABASE'),
    user=os.getenv('PGUSER'),
    password=os.getenv('PGPASSWORD'),
    host=os.getenv('PGHOST'),
    port=os.getenv('PGPORT')
)
print("Database connection established successfully.")

Database connection established successfully.


# Helper — Format Result for Display

Extracts the key fields from an LP result dict into readable DataFrames and printed summaries.

In [4]:
def display_result(result: dict, scenario_label: str = ""):
    """Print a formatted summary of an LP optimization result."""
    if scenario_label:
        print(f"{'='*60}")
        print(f"  SCENARIO: {scenario_label}")
        print(f"{'='*60}")

    # --- Status ---
    cd = result.get("constraint_diagnostics", {})
    status = cd.get("lp_status", "UNKNOWN")
    print(f"\nLP Status: {status}")

    if status not in ("Optimal", "Feasible"):
        reason = cd.get("infeasibility_reason", "No reason provided.")
        print(f"Infeasibility reason: {reason}")
        # Still show excluded suppliers if available
        excl = result.get("excluded_suppliers", [])
        if excl:
            print(f"\nExcluded suppliers ({len(excl)}):")
            print(pd.DataFrame(excl)[["supplier_id", "exclusion_reason"]].to_string(index=False))
        return

    # --- Requirement block ---
    req = result.get("requirement", {})
    print(f"\n--- Requirement ---")
    print(f"  Product:              {result.get('params_recap', {}).get('product', 'N/A')}")
    print(f"  Facility scope:       {result.get('params_recap', {}).get('facility_id', 'all facilities')}")
    print(f"  Total net req (units):{req.get('total_net_requirement', 0):>12,.0f}")
    print(f"  Adjusted req (units): {req.get('adjusted_requirement', 0):>12,.0f}")
    print(f"  Facilities included:  {req.get('n_facilities_included', 0)}")

    fb = req.get("facility_breakdown", [])
    if fb:
        print("\n  Facility breakdown:")
        print(pd.DataFrame(fb).to_string(index=False))

    # --- Supplier pool ---
    sp = result.get("supplier_pool", {})
    print(f"\n--- Supplier Pool ---")
    print(f"  Total suppliers:  {sp.get('n_total_for_product', 0)}")
    print(f"  Eligible:         {sp.get('n_eligible_post_compliance', 0)}")
    print(f"  Selected:         {sp.get('n_selected_by_lp', 0)}")

    # --- Allocation ---
    alloc = result.get("allocation", [])
    if alloc:
        print(f"\n--- Allocation ---")
        alloc_df = pd.DataFrame(alloc)[[
            "supplier_id", "country_code", "decision_tier",
            "allocated_qty", "share_pct",
            "landed_unit_cost", "risk_penalty_norm",
            "total_cost", "moq_met", "bulk_discount_active"
        ]]
        alloc_df["allocated_qty"]   = alloc_df["allocated_qty"].map("{:,.0f}".format)
        alloc_df["share_pct"]       = alloc_df["share_pct"].map("{:.1f}%".format)
        alloc_df["landed_unit_cost"]= alloc_df["landed_unit_cost"].map("${:.4f}".format)
        alloc_df["risk_penalty_norm"]= alloc_df["risk_penalty_norm"].map("{:.3f}".format)
        alloc_df["total_cost"]      = alloc_df["total_cost"].map("${:,.2f}".format)
        print(alloc_df.to_string(index=False))

        # MOQ notes
        moq_notes = [(a["supplier_id"], a["moq_note"]) for a in alloc if a.get("moq_note")]
        if moq_notes:
            print("\n  MOQ / Bulk notes:")
            for sid, note in moq_notes:
                print(f"    {sid}: {note}")

    # --- Cost summary ---
    cs = result.get("cost_summary", {})
    print(f"\n--- Cost Summary ---")
    print(f"  Total cost (USD):         ${cs.get('total_cost_usd', 0):>12,.2f}")
    print(f"  Risk-adjusted cost (USD): ${cs.get('total_risk_adjusted_cost', 0):>12,.2f}")
    print(f"  Avg landed unit cost:     ${cs.get('avg_landed_unit_cost', 0):>12.4f}")
    print(f"  Avg risk penalty (norm):   {cs.get('avg_risk_penalty_norm', 0):>12.3f}")
    budget_util = cs.get("budget_utilization_pct")
    if budget_util is not None:
        print(f"  Budget utilization:        {budget_util:>11.1f}%")
        print(f"  Budget remaining (USD):   ${cs.get('budget_remaining_usd', 0):>12,.2f}")

    # --- Excluded suppliers ---
    excl = result.get("excluded_suppliers", [])
    if excl:
        print(f"\n--- Excluded Suppliers ({len(excl)}) ---")
        print(pd.DataFrame(excl)[["supplier_id", "exclusion_reason"]].to_string(index=False))

    # --- Constraint diagnostics ---
    print(f"\n--- Constraint Diagnostics ---")
    print(f"  Demand fulfilled:          {cd.get('demand_satisfied', False)}")
    print(f"  Total allocated:           {cd.get('total_allocated', 0):>12,.0f}")
    print(f"  Demand floor:              {cd.get('demand_floor', 0):>12,.0f}")
    print(f"  Demand constraint binding: {cd.get('demand_constraint_binding', False)}")
    print(f"  Budget constraint binding: {cd.get('budget_constraint_binding', False)}")
    print(f"  Share constraints binding: {cd.get('n_share_constraints_binding', 0)}")

    # --- Formula description ---
    fd = result.get("formula_description", "")
    if fd:
        print(f"\n--- Formula Description ---")
        print(fd)

    # --- Executive summary ---
    es = result.get("executive_summary", "")
    if es:
        print(f"\n--- Executive Summary ---")
        print(es)

# Identify Available Products

Check which products have positive net procurement requirements (the LP input pool).

In [5]:
req_query = """
    SELECT
        dp.product,
        COUNT(*) AS n_rows,
        SUM(pr.net_requirement)   AS total_net_req,
        COUNT(*) FILTER (WHERE pr.net_requirement > 0) AS rows_with_req
    FROM vw_procurement_requirement pr
    JOIN dim_product dp ON dp.product_key = pr.product_key
    GROUP BY dp.product
    ORDER BY total_net_req DESC
"""
req_df = pd.read_sql(req_query, conn)
print("Products with procurement requirements:")
print(req_df.to_string(index=False))

Products with procurement requirements:
                      product  n_rows  total_net_req  rows_with_req
                  transistors      80     29005.6998             27
                power_devices      80     12815.3562             18
integrated_circuit_components      80      2617.0704              3
              microprocessors      80        42.8843              1


/var/folders/7r/bkznjk192kggm19g_v70_z240000gn/T/ipykernel_98805/3818333495.py:12: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  req_df = pd.read_sql(req_query, conn)


In [6]:
# Pick the product with the highest total net requirement for validation
TEST_PRODUCT = req_df[req_df.rows_with_req > 0].iloc[0]["product"]
print(f"Selected test product: {TEST_PRODUCT}")

Selected test product: transistors


In [7]:
# Identify facilities that have positive requirement for this product
fac_query = """
    SELECT pr.facility_id, SUM(pr.net_requirement) AS facility_net_req
    FROM vw_procurement_requirement pr
    JOIN dim_product dp ON dp.product_key = pr.product_key
    WHERE dp.product = %(product)s
      AND pr.net_requirement > 0
    GROUP BY pr.facility_id
    ORDER BY facility_net_req DESC
"""
fac_df = pd.read_sql(fac_query, conn, params={"product": TEST_PRODUCT})
print(f"Facilities with positive requirement for '{TEST_PRODUCT}':")
print(fac_df.to_string(index=False))

# Save top facility for scenario 5
TEST_FACILITY = fac_df.iloc[0]["facility_id"]
print(f"\nSelected test facility for facility-filter scenario: {TEST_FACILITY}")

/var/folders/7r/bkznjk192kggm19g_v70_z240000gn/T/ipykernel_98805/995137386.py:11: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  fac_df = pd.read_sql(fac_query, conn, params={"product": TEST_PRODUCT})


Facilities with positive requirement for 'transistors':
facility_id  facility_net_req
 FACILITY_3        17368.5990
 FACILITY_4        11637.1008

Selected test facility for facility-filter scenario: FACILITY_3


---
# Scenario 1 — Baseline

**Params:** default lambda_risk=0.5, no budget cap, no share constraint, all facilities, service_level_target=1.0

**Expected:** Optimal, LP selects the cost-efficient / risk-balanced supplier mix.

In [8]:
params_baseline = LPParams(
    product=TEST_PRODUCT,
    lambda_risk=0.50,
    compliance_threshold=0.60,
    service_level_target=1.00,
)

result_baseline = run(params_baseline)
display_result(result_baseline, scenario_label="Baseline (λ=0.50, no constraints)")

/Users/jonathanbeck/Library/CloudStorage/OneDrive-Personal/Desktop/GWU_Spring_26/Business Analytics Capstone/procurement_agent/optimization/run_lp_optimization.py:140: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=qparams)
/Users/jonathanbeck/Library/CloudStorage/OneDrive-Personal/Desktop/GWU_Spring_26/Business Analytics Capstone/procurement_agent/optimization/run_lp_optimization.py:167: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params={'product': params.product})


  SCENARIO: Baseline (λ=0.50, no constraints)

LP Status: Optimal

--- Requirement ---
  Product:              transistors
  Facility scope:       None
  Total net req (units):      29,006
  Adjusted req (units):       29,006
  Facilities included:  2

  Facility breakdown:
facility_id  net_req  share_pct  allocated_qty
 FACILITY_3    17369      59.88          17369
 FACILITY_4    11637      40.12          11637

--- Supplier Pool ---
  Total suppliers:  14
  Eligible:         5
  Selected:         1

--- Allocation ---
supplier_id country_code decision_tier allocated_qty share_pct landed_unit_cost risk_penalty_norm total_cost  moq_met  bulk_discount_active
 SUP_HKG_38          HKG     Preferred        29,006    100.0%          $0.1043             0.276  $3,026.65     True                  True

  MOQ / Bulk notes:
    SUP_HKG_38: MOQ met — bulk pricing applies

--- Cost Summary ---
  Total cost (USD):         $    3,026.65
  Risk-adjusted cost (USD): $    3,444.82
  Avg landed unit co

---
# Scenario 2 — Diversification-Constrained

**Params:** `max_supplier_share=0.40` — no single supplier may fill more than 40% of demand.

**Expected:** Optimal, allocation spread across at least 3 suppliers. Share constraints binding.

In [9]:
params_diversified = LPParams(
    product=TEST_PRODUCT,
    lambda_risk=0.50,
    compliance_threshold=0.60,
    max_supplier_share=0.40,
    service_level_target=1.00,
)

result_diversified = run(params_diversified)
display_result(result_diversified, scenario_label="Diversification-Constrained (max_share=40%)")

  SCENARIO: Diversification-Constrained (max_share=40%)

LP Status: Optimal

--- Requirement ---
  Product:              transistors
  Facility scope:       None
  Total net req (units):      29,006
  Adjusted req (units):       29,006
  Facilities included:  2

  Facility breakdown:
facility_id  net_req  share_pct  allocated_qty
 FACILITY_3    17369      59.88          17369
 FACILITY_4    11637      40.12          11637

--- Supplier Pool ---
  Total suppliers:  14
  Eligible:         5
  Selected:         3

--- Allocation ---
supplier_id country_code decision_tier allocated_qty share_pct landed_unit_cost risk_penalty_norm total_cost  moq_met  bulk_discount_active
 SUP_CAN_10          CAN     Preferred        11,602     40.0%          $0.1062             0.257  $1,231.91     True                  True
 SUP_HKG_38          HKG     Preferred        11,602     40.0%          $0.1043             0.276  $1,210.66     True                  True
 SUP_HKG_35          HKG     Preferred      

/Users/jonathanbeck/Library/CloudStorage/OneDrive-Personal/Desktop/GWU_Spring_26/Business Analytics Capstone/procurement_agent/optimization/run_lp_optimization.py:140: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=qparams)
/Users/jonathanbeck/Library/CloudStorage/OneDrive-Personal/Desktop/GWU_Spring_26/Business Analytics Capstone/procurement_agent/optimization/run_lp_optimization.py:167: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params={'product': params.product})


In [10]:
# Verify: compare baseline vs diversified allocation
if result_baseline["constraint_diagnostics"]["lp_status"] == "Optimal" and \
   result_diversified["constraint_diagnostics"]["lp_status"] == "Optimal":

    base_alloc = pd.DataFrame(result_baseline["allocation"])[["supplier_id", "share_pct"]].rename(
        columns={"share_pct": "baseline_share_pct"})
    div_alloc  = pd.DataFrame(result_diversified["allocation"])[["supplier_id", "share_pct"]].rename(
        columns={"share_pct": "div_share_pct"})

    comparison = base_alloc.merge(div_alloc, on="supplier_id", how="outer").fillna(0)
    comparison["baseline_share_pct"] = comparison["baseline_share_pct"].map("{:.1f}%".format)
    comparison["div_share_pct"]      = comparison["div_share_pct"].map("{:.1f}%".format)
    print("Baseline vs Diversified allocation shares:")
    print(comparison.to_string(index=False))

    max_share = max(a["share_pct"] for a in result_diversified["allocation"])
    print(f"\nMax share in diversified result: {max_share:.1f}% (must be ≤ 40%)")
    assert max_share <= 40.01, f"FAIL: max share {max_share:.2f}% exceeds 40% cap"
    print("PASS: diversification constraint respected.")

Baseline vs Diversified allocation shares:
supplier_id baseline_share_pct div_share_pct
 SUP_CAN_10               0.0%         40.0%
 SUP_HKG_35               0.0%         20.0%
 SUP_HKG_38             100.0%         40.0%

Max share in diversified result: 40.0% (must be ≤ 40%)
PASS: diversification constraint respected.


---
# Scenario 3 — Budget Infeasibility

**Params:** `budget_cap` set to $1 (far below the minimum feasible cost).

**Expected:** Infeasible status, no allocation, clear infeasibility reason in constraint_diagnostics.

In [11]:
params_infeasible = LPParams(
    product=TEST_PRODUCT,
    lambda_risk=0.50,
    compliance_threshold=0.60,
    budget_cap=1.00,          # impossible budget
    service_level_target=1.00,
)

result_infeasible = run(params_infeasible)
display_result(result_infeasible, scenario_label="Budget Infeasibility (budget_cap=$1)")

  SCENARIO: Budget Infeasibility (budget_cap=$1)

LP Status: Infeasible
Infeasibility reason: Check budget vs. demand floor feasibility.

Excluded suppliers (9):
supplier_id     exclusion_reason
 SUP_CHN_15 gate:compliance_gate
 SUP_CHN_17 gate:compliance_gate
 SUP_CHN_18 gate:compliance_gate
 SUP_IDN_42 gate:compliance_gate
 SUP_IDN_43 gate:compliance_gate
 SUP_MEX_54 gate:compliance_gate
 SUP_MEX_55 gate:compliance_gate
 SUP_THA_83 gate:compliance_gate
 SUP_THA_87 gate:compliance_gate


/Users/jonathanbeck/Library/CloudStorage/OneDrive-Personal/Desktop/GWU_Spring_26/Business Analytics Capstone/procurement_agent/optimization/run_lp_optimization.py:140: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=qparams)
/Users/jonathanbeck/Library/CloudStorage/OneDrive-Personal/Desktop/GWU_Spring_26/Business Analytics Capstone/procurement_agent/optimization/run_lp_optimization.py:167: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params={'product': params.product})


In [12]:
# Verify infeasibility is caught correctly
infeasible_status = result_infeasible["constraint_diagnostics"]["lp_status"]
print(f"LP status: {infeasible_status}")
assert infeasible_status not in ("Optimal", "Feasible"), \
    f"FAIL: Expected Infeasible, got {infeasible_status}"

reason = result_infeasible["constraint_diagnostics"].get("infeasibility_reason", "")
print(f"Infeasibility reason: {reason}")
assert reason, "FAIL: infeasibility_reason field is empty"
print("PASS: Infeasibility correctly detected and reported.")

LP status: Infeasible
Infeasibility reason: Check budget vs. demand floor feasibility.
PASS: Infeasibility correctly detected and reported.


---
# Scenario 4 — Service-Level Buffer

**Params:** `service_level_target=1.10` — procure 10% above the base net requirement.

**Expected:** Optimal, `adjusted_requirement` = 1.10 × `total_net_requirement`, total allocated ≥ adjusted requirement.

In [13]:
params_sl = LPParams(
    product=TEST_PRODUCT,
    lambda_risk=0.50,
    compliance_threshold=0.60,
    service_level_target=1.10,
)

result_sl = run(params_sl)
display_result(result_sl, scenario_label="Service-Level Buffer (sl=1.10)")

  SCENARIO: Service-Level Buffer (sl=1.10)

LP Status: Optimal

--- Requirement ---
  Product:              transistors
  Facility scope:       None
  Total net req (units):      29,006
  Adjusted req (units):       31,906
  Facilities included:  2

  Facility breakdown:
facility_id  net_req  share_pct  allocated_qty
 FACILITY_3    17369      59.88          19105
 FACILITY_4    11637      40.12          12801

--- Supplier Pool ---
  Total suppliers:  14
  Eligible:         5
  Selected:         1

--- Allocation ---
supplier_id country_code decision_tier allocated_qty share_pct landed_unit_cost risk_penalty_norm total_cost  moq_met  bulk_discount_active
 SUP_HKG_38          HKG     Preferred        31,906    100.0%          $0.1043             0.276  $3,329.31     True                  True

  MOQ / Bulk notes:
    SUP_HKG_38: MOQ met — bulk pricing applies

--- Cost Summary ---
  Total cost (USD):         $    3,329.31
  Risk-adjusted cost (USD): $    3,789.30
  Avg landed unit cost:

/Users/jonathanbeck/Library/CloudStorage/OneDrive-Personal/Desktop/GWU_Spring_26/Business Analytics Capstone/procurement_agent/optimization/run_lp_optimization.py:140: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=qparams)
/Users/jonathanbeck/Library/CloudStorage/OneDrive-Personal/Desktop/GWU_Spring_26/Business Analytics Capstone/procurement_agent/optimization/run_lp_optimization.py:167: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params={'product': params.product})


In [14]:
# Verify service-level multiplier was applied
if result_sl["constraint_diagnostics"]["lp_status"] == "Optimal":
    base_req = result_sl["requirement"]["total_net_requirement"]
    adj_req  = result_sl["requirement"]["adjusted_requirement"]
    expected = round(base_req * 1.10, 1)

    print(f"Base net requirement:     {base_req:>10,.0f}")
    print(f"Adjusted requirement:     {adj_req:>10,.0f}")
    print(f"Expected (×1.10):         {expected:>10,.0f}")

    # Compare baseline vs sl-buffered total cost
    cost_baseline = result_baseline["cost_summary"].get("total_cost_usd", 0)
    cost_sl       = result_sl["cost_summary"].get("total_cost_usd", 0)
    print(f"\nBaseline total cost: ${cost_baseline:>12,.2f}")
    print(f"SL-buffered cost:    ${cost_sl:>12,.2f}")
    print(f"Cost increase:       ${cost_sl - cost_baseline:>12,.2f} ({(cost_sl/cost_baseline - 1)*100:.1f}%)")

    assert abs(adj_req - expected) < 5, \
        f"FAIL: adjusted_requirement {adj_req} deviates from expected {expected}"
    assert cost_sl > cost_baseline, "FAIL: SL-buffered cost should exceed baseline"
    print("PASS: Service-level buffer correctly increases requirement and cost.")

Base net requirement:         29,006
Adjusted requirement:         31,906
Expected (×1.10):             31,907

Baseline total cost: $    3,026.65
SL-buffered cost:    $    3,329.31
Cost increase:       $      302.66 (10.0%)
PASS: Service-level buffer correctly increases requirement and cost.


---
# Scenario 5 — Facility Filter

**Params:** `facility_id=TEST_FACILITY` — restrict procurement requirement to one facility.

**Expected:** Optimal, `n_facilities_included=1`, smaller `total_net_requirement` than baseline.

In [15]:
params_facility = LPParams(
    product=TEST_PRODUCT,
    facility_id=TEST_FACILITY,
    lambda_risk=0.50,
    compliance_threshold=0.60,
    service_level_target=1.00,
)

result_facility = run(params_facility)
display_result(result_facility, scenario_label=f"Facility Filter ({TEST_FACILITY} only)")

/Users/jonathanbeck/Library/CloudStorage/OneDrive-Personal/Desktop/GWU_Spring_26/Business Analytics Capstone/procurement_agent/optimization/run_lp_optimization.py:140: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params=qparams)


  SCENARIO: Facility Filter (FACILITY_3 only)

LP Status: Optimal

--- Requirement ---
  Product:              transistors
  Facility scope:       FACILITY_3
  Total net req (units):      17,369
  Adjusted req (units):       17,369
  Facilities included:  1

  Facility breakdown:
facility_id  net_req  share_pct  allocated_qty
 FACILITY_3    17369      100.0          17369

--- Supplier Pool ---
  Total suppliers:  14
  Eligible:         5
  Selected:         1

--- Allocation ---
supplier_id country_code decision_tier allocated_qty share_pct landed_unit_cost risk_penalty_norm total_cost  moq_met  bulk_discount_active
 SUP_HKG_38          HKG     Preferred        17,369    100.0%          $0.1043             0.276  $1,812.35     True                  True

  MOQ / Bulk notes:
    SUP_HKG_38: MOQ met — bulk pricing applies

--- Cost Summary ---
  Total cost (USD):         $    1,812.35
  Risk-adjusted cost (USD): $    2,062.76
  Avg landed unit cost:     $      0.1043
  Avg risk penalty 

/Users/jonathanbeck/Library/CloudStorage/OneDrive-Personal/Desktop/GWU_Spring_26/Business Analytics Capstone/procurement_agent/optimization/run_lp_optimization.py:167: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(sql, conn, params={'product': params.product})


In [16]:
# Verify facility filter was applied
if result_facility["constraint_diagnostics"]["lp_status"] == "Optimal":
    n_fac   = result_facility["requirement"]["n_facilities_included"]
    fac_req = result_facility["requirement"]["total_net_requirement"]
    all_req = result_baseline["requirement"]["total_net_requirement"]

    print(f"Facilities in filtered result: {n_fac}")
    print(f"Net requirement (filtered):    {fac_req:>10,.0f}")
    print(f"Net requirement (all fac):     {all_req:>10,.0f}")
    print(f"Facility share of total:       {fac_req/all_req*100:.1f}%")

    assert n_fac == 1, f"FAIL: Expected 1 facility, got {n_fac}"
    assert fac_req <= all_req, "FAIL: Filtered requirement exceeds all-facility baseline"
    print(f"PASS: Facility filter correctly restricts to {TEST_FACILITY}.")

Facilities in filtered result: 1
Net requirement (filtered):        17,369
Net requirement (all fac):         29,006
Facility share of total:       59.9%
PASS: Facility filter correctly restricts to FACILITY_3.


---
# Cross-Scenario Summary

Side-by-side comparison of all five scenarios.

In [17]:
scenarios = [
    ("Baseline",              result_baseline),
    ("Diversified (40%)",     result_diversified),
    ("Budget Infeasible",     result_infeasible),
    ("SL Buffer (×1.10)",     result_sl),
    (f"Facility ({TEST_FACILITY})", result_facility),
]

rows = []
for label, res in scenarios:
    cd  = res.get("constraint_diagnostics", {})
    req = res.get("requirement", {})
    cs  = res.get("cost_summary", {})
    sp  = res.get("supplier_pool", {})
    rows.append({
        "Scenario":        label,
        "Status":          cd.get("lp_status", "N/A"),
        "Net Req":         f"{req.get('total_net_requirement', 0):,.0f}",
        "Adj Req":         f"{req.get('adjusted_requirement', 0):,.0f}",
        "# Selected":      sp.get("n_selected_by_lp", 0),
        "Total Cost": (
            f"${cs['total_cost_usd']:,.2f}" if cs.get("total_cost_usd") else "N/A"
        ),
        "RA Cost": (
            f"${cs['total_risk_adjusted_cost']:,.2f}" if cs.get("total_risk_adjusted_cost") else "N/A"
        ),
        "Budget Util": (
            f"{cs['budget_utilization_pct']:.1f}%"
            if cs.get("budget_utilization_pct") is not None else "—"
        ),
    })

summary_df = pd.DataFrame(rows)
print("Cross-Scenario Summary:")
print(summary_df.to_string(index=False))

Cross-Scenario Summary:
             Scenario     Status Net Req Adj Req  # Selected Total Cost   RA Cost Budget Util
             Baseline    Optimal  29,006  29,006           1  $3,026.65 $3,444.82           —
    Diversified (40%)    Optimal  29,006  29,006           3  $3,053.56 $3,476.39           —
    Budget Infeasible Infeasible  29,006  29,006           0        N/A       N/A        0.0%
    SL Buffer (×1.10)    Optimal  29,006  31,906           1  $3,329.31 $3,789.30           —
Facility (FACILITY_3)    Optimal  17,369  17,369           1  $1,812.35 $2,062.76           —


---
# Executive Summary

Business-readable procurement decision summary for the baseline scenario.

In [18]:
def build_executive_summary(result: dict) -> str:
    """Build a polished, business-readable executive summary from an LP result."""
    cd   = result.get("constraint_diagnostics", {})
    req  = result.get("requirement", {})
    cs   = result.get("cost_summary", {})
    sp   = result.get("supplier_pool", {})
    prec = result.get("params_recap", {})
    alloc = result.get("allocation", [])
    excl  = result.get("excluded_suppliers", [])

    if cd.get("lp_status") not in ("Optimal", "Feasible"):
        reason = cd.get("infeasibility_reason", "No feasible procurement plan found.")
        return (
            f"No feasible procurement plan could be generated for {prec.get('product', 'the selected component')}.\n"
            f"Reason: {reason}\n"
            f"Recommendation: Relax the budget constraint or expand the eligible supplier pool."
        )

    product    = prec.get("product", "the component")
    n_sel      = sp.get("n_selected_by_lp", 0)
    n_elig     = sp.get("n_eligible_post_compliance", 0)
    n_total    = sp.get("n_total_for_product", 0)
    net_req    = req.get("total_net_requirement", 0)
    adj_req    = req.get("adjusted_requirement", 0)
    total_cost = cs.get("total_cost_usd", 0)
    avg_cost   = cs.get("avg_landed_unit_cost", 0)
    avg_risk   = cs.get("avg_risk_penalty_norm", 0)
    lam        = prec.get("lambda_risk", 0.5)
    n_excl     = len(excl)
    n_fac      = req.get("n_facilities_included", 0)

    top_supplier = alloc[0]["supplier_id"] if alloc else "N/A"
    top_share    = alloc[0]["share_pct"] if alloc else 0

    lines = [
        f"PROCUREMENT RECOMMENDATION — {product.upper()}",
        f"{'-'*50}",
        f"",
        f"Procurement target: {adj_req:,.0f} units across {n_fac} facility/facilities.",
        f"",
        f"Supplier selection: {n_sel} of {n_elig} eligible suppliers selected",
        f"({n_total - n_elig} of {n_total} excluded due to compliance threshold or manual exclusion).",
        f"Lead supplier: {top_supplier} at {top_share:.1f}% of total volume.",
        f"",
        f"Expected total procurement cost: ${total_cost:,.2f}",
        f"Average landed unit cost: ${avg_cost:.4f} | Average risk penalty: {avg_risk:.3f} (0=low, 1=high)",
        f"Optimization weight: {int(lam*100)}% risk / {int((1-lam)*100)}% cost.",
    ]

    if n_excl > 0:
        excl_ids = ", ".join(e["supplier_id"] for e in excl[:3])
        if n_excl > 3:
            excl_ids += f" and {n_excl - 3} others"
        lines.append(f"")
        lines.append(f"Excluded suppliers: {excl_ids}.")
        lines.append(f"Primary exclusion reason: {excl[0].get('exclusion_reason', 'N/A')}.")

    return "\n".join(lines)


# Print executive summary for baseline
print(build_executive_summary(result_baseline))

PROCUREMENT RECOMMENDATION — TRANSISTORS
--------------------------------------------------

Procurement target: 29,006 units across 2 facility/facilities.

Supplier selection: 1 of 5 eligible suppliers selected
(9 of 14 excluded due to compliance threshold or manual exclusion).
Lead supplier: SUP_HKG_38 at 100.0% of total volume.

Expected total procurement cost: $3,026.65
Average landed unit cost: $0.1043 | Average risk penalty: 0.276 (0=low, 1=high)
Optimization weight: 50% risk / 50% cost.

Excluded suppliers: SUP_CHN_15, SUP_CHN_17, SUP_CHN_18 and 10 others.
Primary exclusion reason: gate:compliance_gate.


In [19]:
# Also print the formula_description from the LP module itself
print("--- Formula Description (from LP module) ---")
print(result_baseline.get("formula_description", "Not available."))

--- Formula Description (from LP module) ---
Minimize total procurement spend for transistors, weighted 50% cost / 50% risk.

Procurement target: 29,006 units (100% of net requirement = 29,006 units; safety stock coverage already included).
Supplier pool: 5 of 14 transistors suppliers eligible after compliance threshold ≥ 60%.
Facility scope: all facilities with positive net requirement.


---
# Schema Inspection — LP Output Fields

Confirms the full output schema is present and correctly typed. Important for downstream demo agent consumption.

In [20]:
EXPECTED_TOP_KEYS = [
    "params_recap", "requirement", "supplier_pool",
    "allocation", "cost_summary", "excluded_suppliers",
    "constraint_diagnostics", "formula_description"
]

EXPECTED_ALLOC_KEYS = [
    "supplier_id", "country_code", "decision_tier",
    "allocated_qty", "share_pct",
    "landed_unit_cost", "risk_penalty_norm",
    "total_cost", "risk_adjusted_cost_total",
    "top_risk_drivers",
    "bulk_units_threshold", "moq_met",
    "bulk_discount_active", "moq_note"
]

print("Top-level keys check:")
for k in EXPECTED_TOP_KEYS:
    present = k in result_baseline
    print(f"  {'OK' if present else 'MISSING':6s}  {k}")

print("\nAllocation entry keys check (first supplier):")
if result_baseline.get("allocation"):
    first_alloc = result_baseline["allocation"][0]
    for k in EXPECTED_ALLOC_KEYS:
        present = k in first_alloc
        print(f"  {'OK' if present else 'MISSING':6s}  {k}")
else:
    print("  No allocation entries to inspect.")

Top-level keys check:
  OK      params_recap
  OK      requirement
  OK      supplier_pool
  OK      allocation
  OK      cost_summary
  OK      excluded_suppliers
  OK      constraint_diagnostics
  OK      formula_description

Allocation entry keys check (first supplier):
  OK      supplier_id
  OK      country_code
  OK      decision_tier
  OK      allocated_qty
  OK      share_pct
  OK      landed_unit_cost
  OK      risk_penalty_norm
  OK      total_cost
  OK      risk_adjusted_cost_total
  OK      top_risk_drivers
  OK      bulk_units_threshold
  OK      moq_met
  OK      bulk_discount_active
  OK      moq_note


In [21]:
# Final: close DB connection
conn.close()
print("Connection closed. Validation complete.")

Connection closed. Validation complete.
